# PAB-1-Configuration-Chargement-et-Exploration-des-Données

### Configuration et Initialisation de Spark

In [15]:
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .appName("prediction-attrition-bancaire")  
    .master("local[*]")               
    .config("spark.sql.shuffle.partitions", "8")  
    .getOrCreate()
)

print("Spark Version:", spark.version)


Spark Version: 4.0.1


### Chargement des Données

In [16]:
file_path = "../data/dataset-68f599a4c9b84581895311-6907667a3f0be705109390.csv"

df = spark.read.csv(file_path,header=True,inferSchema=True)

df.printSchema()
df.show(5)



root
 |-- RowNumber: integer (nullable = true)
 |-- CustomerId: integer (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: integer (nullable = true)
 |-- IsActiveMember: integer (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)

+---------+----------+--------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId| Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+--------+-----------+---------+------+---+------+---------+-------------+---------+--------------+----

### Statistiques descriptives du DataFrame

In [17]:
df.describe().show()

+-------+------------------+-----------------+-------+-----------------+---------+------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-----------------+-------------------+
|summary|         RowNumber|       CustomerId|Surname|      CreditScore|Geography|Gender|               Age|            Tenure|          Balance|     NumOfProducts|          HasCrCard|     IsActiveMember|  EstimatedSalary|             Exited|
+-------+------------------+-----------------+-------+-----------------+---------+------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-----------------+-------------------+
|  count|             10000|            10000|  10000|            10000|    10000| 10000|             10000|             10000|            10000|             10000|              10000|              10000|            10000|              10000|
|   mean|            5000.5|

###  Nombre de valeurs nulles par colonne

In [18]:
from pyspark.sql.functions import col, sum

# null_count = []

for i in df.columns:
    null_count = df.filter(df[i].isNull()).count()
    print(f"{i}: {null_count} valeurs nulles")

RowNumber: 0 valeurs nulles
CustomerId: 0 valeurs nulles
Surname: 0 valeurs nulles
CreditScore: 0 valeurs nulles
Geography: 0 valeurs nulles
Gender: 0 valeurs nulles
Age: 0 valeurs nulles
Tenure: 0 valeurs nulles
Balance: 0 valeurs nulles
NumOfProducts: 0 valeurs nulles
HasCrCard: 0 valeurs nulles
IsActiveMember: 0 valeurs nulles
EstimatedSalary: 0 valeurs nulles
Exited: 0 valeurs nulles


### Détection des valeurs aberrantes dans les colonnes numériques à l’aide de l’IQR

In [32]:
from pyspark.sql.functions import col

outlier_counts = {} 
numbers_columns = ["CreditScore","Age","Balance","EstimatedSalary"]


for j in numbers_columns:
    q1,q3 = df.approxQuantile(j,[0.25,0.75],0.01)
    
    IQR = q3 - q1
    
    lower = q1-1.5*IQR
    upper = q3+1.5*IQR

    count_outliers = df.filter((col(j) < lower) | (col(j) > upper)).count()
    outlier_counts[j] = count_outliers
    
for col_name, count in outlier_counts.items():
    print(f"Colonne '{col_name}' : {count} outliers")


Colonne 'CreditScore' : 17 outliers
Colonne 'Age' : 359 outliers
Colonne 'Balance' : 0 outliers
Colonne 'EstimatedSalary' : 0 outliers
